# DS 227 · Week 5 — Data Preprocessing I
## Cleaning, missing data and outliers

Every code block from the week 5 lecture, in order, so you can run what you watched
instead of retyping it. The data is fetched from the course API, so there is no file to
download and nothing to unzip.

**How to use this.** Run the cells top to bottom the first time. After that, change things:
the point of a notebook is that you can break it and find out why.

A section at the end, **Beyond the lecture**, goes further than the slides did. It is the
analysis the lecture did not have time for, and it contains the single most important
result in this dataset.

---

### Setup

One cell, run once. `requests` fetches the data; `io` hands it to pandas as if it were a
file.

> **Why `requests` and not `pd.read_csv(url)`?** pandas uses `urllib` internally, and this
> site's bot protection answers `urllib`'s default User-Agent with **403 Forbidden**. The
> request below sets a normal one. This is a real trap, not a style preference — you will
> hit it again on any site behind a CDN.


In [ ]:
import pandas as pd
import numpy as np
import requests, io

URL = "https://portal.latarak.com/api/flood/export.csv"

r = requests.get(URL, timeout=60)
r.raise_for_status()                       # fail loudly now, not three cells later
flood = pd.read_csv(io.StringIO(r.text))   # keep a pristine copy to restore from

df = flood.copy()
print(df.shape)                            # (34079, 15)

---
## Part 1 — The lecture, cell by cell

Headings match the slide titles, so you can jump to whichever one you want to replay.


### Slide 9 — Counting missing values per column


Expected output:

```
city       0
temp       2
rain_mm    1
dtype: int64
city  temp  rain_mm
1     Davao   NaN      4.0
2    Iloilo  33.0      NaN
4  Tacloban   NaN      0.0
```


In [ ]:
df = pd.DataFrame({"city": ["Cebu", "Davao", "Iloilo", "Baguio", "Tacloban"],
                   "temp": [31, np.nan, 33, 29, np.nan],
                   "rain_mm": [12, 4, np.nan, 8, 0]})
print(df.isna().sum())


# and the actual damaged rows
print(df[df.isna().any(axis=1)])

### Slide 10 — The flood control dataset before cleaning


Expected output:

```
(34079, 15)
```


In [ ]:
import pandas as pd, requests, io

url = "https://portal.latarak.com/api/flood/export.csv"
r   = requests.get(url, timeout=60)
df  = pd.read_csv(io.StringIO(r.text))
df.shape


# 15 columns, and not one of them is clean

### Slide 11 — Counting missing values in 34,079 rows


Expected output:

```
completion_date    6414
longitude          3195
latitude           3195
start_date          973
contractor          876
description           0
contract_id           0
```


In [ ]:
df.isna().sum().sort_values(ascending=False)

# … the other 8 columns are 0 too
# 18.8% of completion_date is empty

### Slide 13 — Cross-tabulating missingness against a reliable column


Expected output:

```
status
Completed          0.0
For Procurement    1.0
Not Yet Started    1.0
On-Going           1.0
Terminated         0.0
Name: completion_date, dtype: float64
```


In [ ]:
df.groupby("status")["completion_date"].apply(
    lambda s: s.isna().mean())

### Slide 15 — Three fields missing before contract award


Expected output:

```
876
876
876
876
```


In [ ]:
fp = df[df.status == "For Procurement"]
len(fp)


fp.contractor.isna().sum()

(fp.budget == 0).sum()

fp.completion_date.isna().sum()


# three columns, one cause

### Slide 17 — Dropping rows: no invented values, but less data


Expected output:

```
region  temp
0  Visayas  31.0
2  Visayas  33.0
3    Luzon  29.0    ← 5 rows in, 3 rows out
```


In [ ]:
df = pd.DataFrame(
    {"region": ["Visayas", "Luzon", "Visayas", "Luzon", "Visayas"],
     "temp":   [31, np.nan, 33, 29, np.nan]})

df.dropna()

### Slide 18 — Filling gaps: rows kept, values invented


Expected output:

```
region  temp
0  Visayas  31.0
1    Luzon  31.0   ← invented
2  Visayas  33.0
3    Luzon  29.0
4  Visayas  31.0   ← invented
```


In [ ]:
df.fillna(df["temp"].mean())

### Slide 19 — Mean, median or mode


Expected output:

```
31.4
12.0
12.0
```


In [ ]:
rain = pd.Series([12, 45, np.nan, 80, 8, 12])

print(round(rain.mean(), 1))
print(rain.median())
print(rain.mode()[0])

### Slide 20 — Filling with group means


Expected output:

```
region  temp
0  Visayas  31.0
1    Luzon  29.0   ← Luzon mean
2  Visayas  33.0
3    Luzon  29.0
4  Visayas  32.0   ← Visayas mean
```


In [ ]:
df["temp"] = df["temp"].fillna(
    df.groupby("region")["temp"]
      .transform("mean"))
df

> **Back to the real data.** The cells above rebound `df` to the small worked example.
> This puts the 34,079-row frame back, so the rest of the lecture runs on it.


In [ ]:
df = flood.copy()
print(df.shape)

### Slide 22 — One method call removes 8,709 rows


Expected output:

```
(34079, 25370)
8709          # 25.6% dropped
```


In [ ]:
clean = df.dropna()

len(df), len(clean)


34079 - 25370

### Slide 23 — Which rows the whole-frame drop removed


Expected output:

```
status
Completed          27534
On-Going            5442
For Procurement      876
Terminated           131
Not Yet Started       96
Name: count, dtype: int64
```


In [ ]:
df.status.value_counts()

### Slide 23 — Which rows the whole-frame drop removed


Expected output:

```
status
Completed          25274
Terminated            96
Name: count, dtype: int64

# On-Going:        GONE
# For Procurement: GONE
# Not Yet Started: GONE
```


In [ ]:
clean.status.value_counts()

### Slide 24 — After dropping, 99.6% of projects appear complete


Expected output:

```
0.996         # "99.6% completed!"
0.808         # 80.8%
```


In [ ]:
round(25274 / len(clean), 3)


# the real figure, before cleaning
round(27534 / len(df), 3)


# you moved it 19 points by typing
# six characters

### Slide 29 — Duplicates: the same row counted twice


Expected output:

```
[False, False, True]
2
0
0
```


In [ ]:
t = pd.DataFrame({"city": ["Cebu", "Davao", "Cebu"],
                  "temp": [31, 30, 31]})
print(t.duplicated().tolist())
print(len(t.drop_duplicates()))



# the flood data
print(df.duplicated().sum())
print(df.contract_id.duplicated().sum())

### Slide 30 — Every column has a data type


Expected output:

```
budget        float64
year            int64
start_date     object
status         object
dtype: object
```


In [ ]:
print(df[["budget", "year", "start_date", "status"]].dtypes)

### Slide 31 — When a number column contains “n/a”


In [ ]:
import pandas as pd
raw = pd.Series(["31", "n/a", "33"])
nums = pd.to_numeric(raw)

The cell above **fails on purpose**:

```
Traceback (most recent call last):
  File "clean.py", line 3, in <module>
    nums = pd.to_numeric(raw)
  … pandas’ own inner lines (trimmed) …
ValueError: Unable to parse string "n/a" at position 1
```

`errors="coerce"` turns what it cannot parse into `NaN` instead of raising:


In [ ]:
nums = pd.to_numeric(raw, errors="coerce")
print(nums.tolist())

### Slide 34 — An outlier is a point far from the rest


In [ ]:
s = pd.Series(
    [12, 14, 13, 15, 11, 14, 98, 13, 12])

### Slide 35 — The IQR rule and its fences


Expected output:

```
6    98       ← flagged
dtype: int64
```


In [ ]:
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
low  = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

s[(s < low) | (s > high)]

### Slide 37 — A z-score counts standard deviations


Expected output:

```
22.44 28.36
[-0.37, -0.3, -0.33, -0.26, -0.4, -0.3, 2.66, -0.33, -0.37]
[98]
```


In [ ]:
z = (s - s.mean()) / s.std()

print(round(s.mean(), 2), round(s.std(), 2))
print(z.round(2).tolist())
print(s[z.abs() > 2].tolist())

### Slide 40 — The mean is sensitive to outliers; the median is not


Expected output:

```
22.44     ← dragged up by the 98
13.0      ← what a typical value looks like
```


In [ ]:
round(s.mean(), 2)


s.median()

### Slide 41 — Distribution of the budget column


Expected output:

```
count          34,079
mean       46,952,340
std        48,594,319
min                 0
25%        14,649,406
50%        37,676,233
75%        67,549,270
max     1,447,499,996
Name: budget, dtype: float64
```


In [ ]:
pd.set_option("display.float_format", "{:,.0f}".format)
print(df.budget.describe())

# max is 38x the median

### Slide 42 — IQR applied to 34,079 real contracts


Expected output:

```
14649406 67549270 52899864 146899067
798
```


In [ ]:
q1, q3 = df.budget.quantile([.25, .75])
iqr = q3 - q1
fence = q3 + 1.5 * iqr

print(round(q1), round(q3), round(iqr), round(fence))


(df.budget > fence).sum()

### Slide 44 — Mean and median budgets differ by 25%


Expected output:

```
46952340
37676233
1.25          # mean is 25% higher
```


In [ ]:
round(df.budget.mean())

round(df.budget.median())


round(46952340 / 37676233, 2)

### Slide 45 — 939 projects with a budget of exactly zero


Expected output:

```
939
status
For Procurement    876
Terminated          63
Name: count, dtype: int64
```


In [ ]:
z = df[df.budget == 0]
len(z)


z.status.value_counts()


# 0 here means "not applicable",
# NOT "free"

### Slide 53 — A cleaning log kept in code


In [ ]:
# cleaning log — keep it in the notebook
log = []

def step(name, before, after, why):
    log.append({"step": name,
                "dropped": before - after,
                "why": why})

step("no budget", 34079, 33140,
     "cannot answer a spend question")

---
# Part 2 — Beyond the lecture

Everything below is new. The slides stopped at *how* to find missing values and outliers;
this asks *what this particular dataset is actually telling you*, which is the part that
decides whether an analysis is honest.

Five questions, in the order a careful analyst would ask them.


## 1. Is `completion_date` missing at random?

The lecture found 6,414 missing completion dates and moved on. That is the moment most
analyses go wrong, because the obvious next step — `dropna()` — is only safe if the gaps
are **random**. Cross-tabulate the gap against `status` and see.


In [ ]:
gaps = (df.groupby("status")["completion_date"]
          .agg(projects="size", missing=lambda s: s.isna().sum()))
gaps["missing_%"] = (100 * gaps.missing / gaps.projects).round(1)
gaps.sort_values("projects", ascending=False)

**Read that table before continuing.** The pattern is not noise:

| status | missing |
|---|---|
| Completed | **0%** |
| Terminated | **0%** |
| On-Going | **100%** |
| For Procurement | **100%** |
| Not Yet Started | **100%** |

The value is missing **exactly when the project has not finished**. Nothing was lost in
collection — an unfinished project has no completion date to record. The gap is not a
defect in the data; it *is* data.

So `df.dropna()` does not remove bad rows here. It removes **every unfinished project**,
which is why the lecture's cleaned frame came out 99.6% "Completed". You would then report
on a dataset from which all in-progress work had been silently deleted.

> **The rule.** Before dropping rows for a missing value, cross-tabulate that column
> against a column you trust. If the missingness has a pattern, dropping encodes that
> pattern as a bias.


In [ ]:
# What dropping actually costs, stated plainly.
before = df.status.value_counts(normalize=True).mul(100).round(1)
after  = df.dropna().status.value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({"before %": before, "after dropna() %": after}).fillna(0)

## 2. The `amount_paid` column

A column of numbers that is present on every row, never null, and completely unusable.


In [ ]:
print("rows where amount_paid is not zero:", int((df.amount_paid.fillna(0) != 0).sum()))
print("total budget      : PHP %.2f trillion" % (df.budget.sum() / 1e12))
print("budget - amount_paid: PHP %.2f trillion" % (
      (df.budget.fillna(0) - df.amount_paid.fillna(0)).sum() / 1e12))

`amount_paid` is **0 on all 34,079 rows**, and on all 248,220 rows of the upstream
export too. The field is simply not populated.

Subtract it from `budget` and you get PHP 1.60 trillion of apparently unpaid public works,
against named, real contractors. That number is **false**, and publishing it would be a
serious accusation built on an empty column.

> A column being present is not evidence that it is populated. `.describe()` would have
> shown `max = 0` in one line — run it on new data before you trust any column.


In [ ]:
df[["budget", "amount_paid", "progress"]].describe()

## 3. Duplicates

Worth two lines, because finding none is also a result.


In [ ]:
print("duplicate contract_id :", int(df.contract_id.duplicated().sum()))
print("fully identical rows  :", int(df.duplicated().sum()))

Zero and zero. `contract_id` is a genuine key, so joins against it are safe — which
is the kind of thing worth checking *before* you build an analysis that assumes it.


## 4. Outliers: two rules, two different answers

The lecture applied the IQR rule. Compare it against the z-score rule on the same column
and notice they disagree.


In [ ]:
b = df.budget.dropna()
q1, q3 = b.quantile([.25, .75])
iqr    = q3 - q1
fence  = q3 + 1.5 * iqr
z      = (b - b.mean()) / b.std()

print(f"median           PHP {b.median():,.0f}")
print(f"IQR upper fence  PHP {fence:,.0f}  -> {(b > fence).sum():,} rows ({100*(b>fence).mean():.1f}%)")
print(f"|z| > 3                            -> {(z.abs() > 3).sum():,} rows ({100*(z.abs()>3).mean():.2f}%)")
print(f"largest budget   PHP {b.max():,.0f}")

The IQR rule flags **798** rows; the z-score rule flags **468**. Same column, same
day, different answer — because the z-score uses the mean and standard deviation, and both
of those are themselves dragged upward by the very values being tested.

Now the question that matters: **is a PHP 1.45 billion flood control contract an error?**

No. It is a large project. Neither rule can tell you that, because neither knows what the
numbers mean. An "outlier" is a statistical statement about distance from the centre; it
is not a synonym for "wrong". Deleting these would delete the biggest infrastructure in
the country from an analysis of infrastructure spending.


In [ ]:
# Look before deleting. These are the ten largest contracts.
(df.nlargest(10, "budget")[["contract_id", "region", "budget", "status"]]
   .assign(budget=lambda d: d.budget.map("{:,.0f}".format)))

## 5. What the geography tells you

`latitude` and `longitude` are each missing on 3,195 rows. Same question as section 1 —
is it random?


In [ ]:
geo = (df.groupby("region")["latitude"]
         .agg(projects="size", missing=lambda s: s.isna().sum()))
geo["missing_%"] = (100 * geo.missing / geo.projects).round(1)
geo[geo.projects > 200].sort_values("missing_%", ascending=False).head(8)

Not random either: it ranges from about 17.5% in Region IV-B down to roughly 12% in
others. Map these projects and some regions quietly lose more of their work than others —
a map that is wrong in a way that looks fine.

---

## Putting it together

Having checked all of the above, a defensible spend-by-region figure drops rows only for
the column being measured — not with a blanket `dropna()`.


In [ ]:
spend = (df.dropna(subset=["budget"])          # only the column we are summing
           .groupby("region")["budget"].sum()
           .sort_values(ascending=False) / 1e9)
spend.head(10).round(1).rename("PHP billions")

### The cleaning log

Record what you removed and why. If you cannot justify a step in one line, it is not a
cleaning step — it is a change to the result.


In [ ]:
log = []

def step(name, before, after, why):
    log.append({"step": name, "rows_dropped": before - after, "why": why})

n0 = len(df)
n1 = len(df.dropna(subset=["budget"]))
step("drop rows with no budget", n0, n1, "cannot be summed in a spend question")

pd.DataFrame(log)

---

## What to take from this

1. **A missing value is not automatically a defect.** `completion_date` is absent because
   the project is unfinished. Dropping those rows deletes every in-progress project.
2. **A populated column is not automatically usable.** `amount_paid` is zero everywhere,
   and trusting it manufactures a PHP 1.6 trillion accusation.
3. **An outlier is not automatically an error.** The largest contracts are real, and the
   two standard rules disagree about which rows even qualify.
4. **Check whether missingness has a pattern** — against `status`, against `region` —
   before you remove anything.
5. **Keep the log.** Cleaning that nobody can reconstruct is indistinguishable from
   fabrication.

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at*
`/api/flood` *— see* `/api/flood` *itself for the full endpoint list.*
